<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week04-antennas-propagation.ipynb)

# Week 4 — Antennas and Propagation
**ECE 512 Wireless Communications** · companion notebook to the Week 4 lecture

This notebook lets you experiment with the ideas from the slides:

1. Far field and the Fraunhofer (Rayleigh) distance $R_{\mathrm{ff}} = 2D^2/\lambda$
2. Radiation intensity and radiation patterns (small dipole, half-wave dipole)
3. Directivity, gain and half-power beamwidth
4. Antenna arrays and beam steering (array factor, grating lobes)
5. Reflection: Fresnel coefficients and the Brewster angle
6. The two-ray ground-reflection model and the $d^{-4}$ region
7. Knife-edge diffraction and Fresnel zones

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, FloatLogSlider, Dropdown
from ece512 import antennas as ant
from ece512.units import wavelength, linear_to_db

## 1. Far field and the Fraunhofer distance
Far from the antenna every radiated field has the form
$$\mathbf{E}(r,\theta,\phi) = \left[\hat{\theta}\,F_\theta(\theta,\phi) + \hat{\phi}\,F_\phi(\theta,\phi)\right]\frac{e^{-jk_0 r}}{r},\qquad k_0 = \frac{2\pi}{\lambda},$$
i.e. the *shape* of the pattern no longer depends on $r$. The spherical wavefront looks locally plane once
the path-length difference across the antenna aperture $D$ is small. Requiring the phase error on
boresight, $k_0\left(\sqrt{r^2 + (D/2)^2} - r\right) \approx \dfrac{\pi D^2}{4\lambda r}$, to stay below $\pi/8$ (22.5°) gives
$$R_{\mathrm{ff}} \approx \frac{2D^2}{\lambda}\quad\text{(electrically large)},\qquad R_{\mathrm{ff}} \approx 2\lambda\quad\text{(electrically small)}.$$
First the two examples from the slides:

In [ ]:
examples = [("18-in satellite dish, 12.4 GHz", 0.457, 12.4e9),
            ("1 m antenna, 900 MHz", 1.0, 900e6),
            ("half-wave dipole, 900 MHz", 0.5 * wavelength(900e6), 900e6),
            ("8x8 patch array (4 cm), 28 GHz", 0.04, 28e9),
            ("64-element array (1 m), 3.5 GHz", 1.0, 3.5e9)]
print(f"{'antenna':<34}{'lambda (cm)':>12}{'D/lambda':>10}{'R_ff (m)':>10}{'phase err at R_ff':>19}")
for name, D, f in examples:
    lam = wavelength(f)
    rff = ant.far_field_distance(D, lam)
    err = np.degrees(ant.aperture_phase_error(D, rff, lam))
    print(f"{name:<34}{100 * lam:12.2f}{D / lam:10.2f}{rff:10.2f}{err:16.1f} deg")

The slide values (17.3 m and 6 m) use $c = 3\times10^8$ m/s; the table uses the exact $c$.
For the dipole $2D^2/\lambda = \lambda/2 < 2\lambda$, so the small-antenna rule sets the far field.

**Near field vs. far field.** Below we sum exact spherical waves $e^{-jk_0R}/R$ from a uniform line source of
length $D$ observed at distance $r$, and compare with the far-field pattern
$|\mathrm{sinc}(k_0 D\cos\theta/2)|$. Slide $r/R_{\mathrm{ff}}$ down: the nulls fill in and the sidelobes merge into
the main beam. At $r = R_{\mathrm{ff}}$ the pattern is already very close to its far-field shape.

In [ ]:
def show_near_field(D_over_lambda=10.0, r_over_rff=0.1):
    lam = 1.0
    D = D_over_lambda * lam
    rff = ant.fraunhofer_distance(D, lam)
    span = min(90.0, np.degrees(6 / D_over_lambda))       # show a few sidelobes
    off = np.radians(np.linspace(-span, span, 1201))      # angle from broadside
    theta = np.pi / 2 - off
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.degrees(off), linear_to_db(ant.line_source_field(theta, D, lam) ** 2 + 1e-12),
            "k--", lw=1.5, label="far field (sinc)")
    for frac, c in [(r_over_rff, "tab:red"), (1.0, "tab:blue")]:
        f = ant.line_source_field(theta, D, lam, r_m=frac * rff)
        ax.plot(np.degrees(off), linear_to_db(f**2 + 1e-12), color=c, lw=2 if c == "tab:red" else 1.2,
                label=rf"exact at $r = {frac:.2g}\,R_{{\mathrm{{ff}}}}$ = {frac * rff:.1f}$\lambda$")
    ax.set_ylim(-40, 1); ax.set_xlim(-span, span)
    ax.set_xlabel("angle from broadside (deg)"); ax.set_ylabel("normalized power pattern (dB)")
    ax.set_title(rf"Uniform line source, $D = {D_over_lambda:g}\lambda$, $R_{{\mathrm{{ff}}}} = {rff:.0f}\lambda$")
    ax.grid(alpha=0.3); ax.legend(loc="upper right", fontsize=8, framealpha=0.95); plt.show()

interact(show_near_field, D_over_lambda=FloatSlider(value=10, min=2, max=20, step=1), r_over_rff=FloatLogSlider(value=0.1, base=10, min=-2, max=1, step=0.1));

## 2. Radiation intensity and radiation patterns
The radiation intensity (power per unit solid angle) is
$$U(\theta,\phi) = \frac{1}{2\eta_0}\left[|F_\theta|^2 + |F_\phi|^2\right],\qquad
P_{\mathrm{rad}} = \int_0^{2\pi}\!\!\int_0^{\pi} U(\theta,\phi)\sin\theta\,d\theta\,d\phi .$$
For antennas along the $z$ axis:

| Antenna | $F_\theta(\theta)$ (normalized) |
|---|---|
| small (Hertzian) dipole | $\sin\theta$ |
| dipole of length $L$ | $\dfrac{\cos\!\left(\frac{k_0L}{2}\cos\theta\right) - \cos\frac{k_0L}{2}}{\sin\theta}$ |
| half-wave dipole ($L=\lambda/2$) | $\dfrac{\cos\!\left(\frac{\pi}{2}\cos\theta\right)}{\sin\theta}$ |

None of these depend on $\phi$: they are **omnidirectional in azimuth**. The elevation pattern plots
$10\log_{10} U$ versus $\theta$.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 1441)     # full circle for the elevation-plane cut
pats = {"small dipole": ant.small_dipole_field(theta),
        r"half-wave dipole": ant.half_wave_dipole_field(theta),
        r"$L = 1.25\lambda$ dipole": ant.dipole_field(theta, 1.25)}
fig = plt.figure(figsize=(12, 4.3))
ax1 = fig.add_subplot(1, 3, 1, projection="polar")
ax2 = fig.add_subplot(1, 3, 2, projection="polar")
ax3 = fig.add_subplot(1, 3, 3, projection="polar")
for (name, f), c in zip(pats.items(), ["tab:blue", "tab:orange", "tab:green"]):
    ax1.plot(theta, f**2, color=c, label=name)
    ax2.plot(theta, np.maximum(linear_to_db(f**2 + 1e-12), -30), color=c)
ax3.plot(theta, np.ones_like(theta), color="tab:blue", label="all three (no $\\phi$ dependence)")
for ax in (ax1, ax2):
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)   # theta measured from +z (up)
ax1.set_title("elevation cut, linear $U/U_{\\max}$", pad=18); ax1.set_rticks([0.25, 0.5, 0.75, 1])
ax2.set_title("elevation cut, dB (floor $-30$ dB)", pad=18)
ax2.set_rticks([-30, -20, -10, 0]); ax2.set_rlim(-30, 0)
for ax in (ax1, ax2):
    ax.set_rlabel_position(22.5); ax.tick_params(axis="y", labelsize=8)
ax3.set_title("azimuth cut ($\\theta = 90^\\circ$)", pad=18); ax3.set_rticks([0.5, 1])
fig.legend(*ax1.get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

The same patterns in 3D (radius and color $= |F(\theta,\phi)|$, antenna axis along $z$), plus an 8-element
broadside array (Section 4) to show how an array squeezes the doughnut into a thin disk.

In [ ]:
th, ph = np.meshgrid(np.linspace(0, np.pi, 91), np.linspace(0, 2 * np.pi, 121), indexing="ij")
shapes3d = {"small dipole": ant.small_dipole_field(th),
            "half-wave dipole": ant.half_wave_dipole_field(th),
            r"$L = 1.25\lambda$ dipole": ant.dipole_field(th, 1.25),
            r"8-element array, $d=\lambda/2$": ant.array_factor(th, 8, 0.5)}
fig = plt.figure(figsize=(14, 4))
for k, (name, r) in enumerate(shapes3d.items()):
    ax = fig.add_subplot(1, 4, k + 1, projection="3d")
    X, Y, Z = r * np.sin(th) * np.cos(ph), r * np.sin(th) * np.sin(ph), r * np.cos(th)
    ax.plot_surface(X, Y, Z, facecolors=plt.cm.viridis(r), rstride=1, cstride=1, linewidth=0, shade=False)
    ax.view_init(elev=18, azim=-55)
    ax.set_box_aspect((1, 1, 1)); ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_xlabel("x", labelpad=-12); ax.set_ylabel("y", labelpad=-12); ax.set_zlabel("z", labelpad=-12)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 3. Directivity, gain and half-power beamwidth
$$D = \frac{4\pi\,U_{\max}}{\displaystyle\int_{0}^{\pi}\!\!\int_{0}^{2\pi} U(\theta,\phi)\sin\theta\,d\phi\,d\theta},
\qquad D_{\mathrm{dB}} = 10\log_{10} D \ \ \text{(dBi)},\qquad G = \epsilon_{\mathrm{rad}} D .$$
For the small dipole the slides evaluate the integral in closed form:
$D = 4\pi C / \left(\tfrac{8\pi}{3}C\right) = 3/2 = 1.76$ dBi. Below we integrate $U$ numerically
(trapezoidal rule on a $\theta,\phi$ grid) and compare with the lecture's table. The **half-power
beamwidth (HPBW)** is the angular width of the main lobe between the points where $U$ drops to $U_{\max}/2$ ($-3$ dB).

In [ ]:
cases = [("isotropic",          lambda t, p: np.ones_like(t),                       1.0,  None),
         ("small dipole",       lambda t, p: ant.small_dipole_field(t) ** 2,         1.5,  ant.small_dipole_field),
         ("half-wave dipole",   lambda t, p: ant.half_wave_dipole_field(t) ** 2,     1.64, ant.half_wave_dipole_field),
         ("full-wave dipole",   lambda t, p: ant.dipole_field(t, 1.0) ** 2,          2.41, lambda t: ant.dipole_field(t, 1.0)),
         ("U = cos^2(theta), upper hemisphere",
                                lambda t, p: np.where(t <= np.pi / 2, np.cos(t) ** 2, 0.0), 6.0, None)]
print(f"{'pattern':<36}{'D numeric':>10}{'D (dBi)':>9}{'closed form / slide':>21}{'HPBW':>9}")
for name, U, ref, F in cases:
    D = ant.directivity(U)
    hp = f"{np.degrees(ant.hpbw(F)):6.1f} deg" if F is not None else "      -"
    print(f"{name:<36}{D:10.4f}{10 * np.log10(D):9.2f}{(f'{ref:g}' if ref else '-'):>21}  {hp}")

The half-wave dipole's exact directivity is 1.641 (2.15 dBi); the slide table rounds it to 1.64 (the full-wave
reference value 2.41 is from Balanis). Note
that $\cos^2\theta$ over a hemisphere (a crude patch-like pattern) already reaches 6 = 7.8 dBi, the top of the
"patch" range in the slide table. A useful rule of thumb connects the two quantities: narrower beams mean
higher directivity, $D \approx 41\,000/(\mathrm{HPBW}_E\,\mathrm{HPBW}_H)$ with beamwidths in degrees.

**Explore:** change the dipole length. Directivity peaks near $L \approx 1.25\lambda$; beyond that the
main beam splits into multiple lobes and the broadside maximum collapses (at $L = 2\lambda$ it is a null).

In [ ]:
def show_dipole(L_over_lambda=0.5, efficiency=0.9):
    t = np.linspace(0, 2 * np.pi, 1441)
    f = ant.dipole_field(t, L_over_lambda)
    D = ant.directivity(lambda th, ph: ant.dipole_field(th, L_over_lambda) ** 2, n_phi=3)
    G = efficiency * D
    fig = plt.figure(figsize=(11, 4))
    ax = fig.add_subplot(1, 2, 1, projection="polar")
    ax.plot(t, f**2, color="tab:orange", label=rf"$L = {L_over_lambda:.2f}\lambda$")
    ax.plot(t, ant.half_wave_dipole_field(t) ** 2, "k--", lw=1, label=r"$\lambda/2$ reference")
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.set_rticks([0.5, 1])
    ax.set_title(f"elevation pattern $U/U_{{\\max}}$\nD = {D:.2f} = {10 * np.log10(D):.2f} dBi,  "
                 f"G = {10 * np.log10(G):.2f} dBi", fontsize=10)
    ax.legend(loc="lower left", bbox_to_anchor=(-0.25, -0.15), fontsize=8)
    ax2 = fig.add_subplot(1, 2, 2)
    Ls = np.linspace(0.05, 2.0, 80)
    Ds = [ant.directivity(lambda th, ph, L=L: ant.dipole_field(th, L) ** 2, n_phi=3) for L in Ls]
    ax2.plot(Ls, 10 * np.log10(Ds), color="tab:blue")
    ax2.plot(L_over_lambda, 10 * np.log10(D), "o", color="tab:orange", ms=8)
    ax2.axhline(1.76, color="gray", ls=":", lw=1); ax2.text(1.55, 1.9, "small dipole 1.76 dBi", fontsize=8, color="gray")
    ax2.set_xlabel(r"dipole length $L/\lambda$"); ax2.set_ylabel("directivity (dBi)")
    ax2.set_title("directivity vs. length (max direction)"); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_dipole, L_over_lambda=FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05), efficiency=FloatSlider(value=0.9, min=0.3, max=1.0, step=0.05));

## 4. Antenna arrays and beam steering
The beamsteering antennas in the lecture (60 GHz arrays, metamaterial leaky-wave antennas, RIS panels) all
rely on the same idea: many elements whose signals add in phase in one direction. For $N$ identical
elements on the $z$ axis with spacing $d$ and progressive phase $\beta$,
$$\psi = k_0 d\cos\theta + \beta,\qquad |\mathrm{AF}(\theta)| = \left|\frac{\sin(N\psi/2)}{N\sin(\psi/2)}\right|,$$
and the total pattern is (element pattern) $\times$ (array factor). The main beam sits where $\psi = 0$, so
choosing $\beta = -k_0 d\cos\theta_0$ **steers** it to $\theta_0$ ($\theta_0 = 90^\circ$ is broadside).
Additional full-strength maxima (**grating lobes**) appear wherever $\psi = 2\pi m$ inside visible space;
they are avoided when $d/\lambda < 1/(1 + |\cos\theta_0|)$, i.e. $d < \lambda/2$ guarantees none at any steering angle.

**Explore:** increase $d/\lambda$ past 0.5 while steering off broadside, and watch the grating lobe (red markers)
enter visible space.

In [ ]:
def show_array(N=8, d_over_lambda=0.5, steer_deg=90.0, element="isotropic"):
    t = np.linspace(0, np.pi, 3601)
    th0 = np.radians(steer_deg)
    beta = ant.steering_phase(th0, d_over_lambda)
    af = ant.array_factor(t, N, d_over_lambda, beta)
    elem = {"isotropic": lambda x: np.ones_like(x), "half-wave dipole (along z)": ant.half_wave_dipole_field,
            "small dipole (along z)": ant.small_dipole_field}[element]
    total = af * elem(t)
    D = ant.directivity(lambda th, ph: (ant.array_factor(th, N, d_over_lambda, beta) * elem(th)) ** 2,
                        n_theta=3601, n_phi=3)
    # predicted grating-lobe directions: cos(theta_g) = cos(theta_0) + m / (d/lambda), |.| <= 1
    m = np.arange(-6, 7); m = m[m != 0]
    cg = np.cos(th0) + m / d_over_lambda
    grating = np.degrees(np.arccos(cg[np.abs(cg) <= 1]))
    fig = plt.figure(figsize=(12, 4.2))
    ax = fig.add_subplot(1, 2, 1, projection="polar")
    tt = np.concatenate([t, 2 * np.pi - t[::-1]])                 # mirror: pattern is symmetric about z
    ax.plot(tt, np.concatenate([total, total[::-1]]), color="tab:blue")
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.set_rticks([0.5, 1])
    ax.set_title(r"$|F\cdot\mathrm{AF}|$ (elevation, $\theta$ from array axis $z$)", fontsize=10, pad=14)
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot(np.degrees(t), linear_to_db(total**2 + 1e-12), color="tab:blue", label="total pattern")
    ax2.axvline(steer_deg, color="k", ls="--", lw=1, label=rf"steering $\theta_0$ = {steer_deg:.0f}°")
    for k, g in enumerate(grating):
        ax2.plot(g, 0, "v", color="tab:red", ms=9, label="predicted grating lobe" if k == 0 else None)
    hp = np.degrees(ant.hpbw(total, t))
    ax2.set_ylim(-40, 3); ax2.set_xlim(0, 180)
    ax2.set_xlabel(r"$\theta$ (deg)"); ax2.set_ylabel("normalized power (dB)")
    ax2.set_title(rf"N = {N}, d = {d_over_lambda:.2f}$\lambda$, $\beta$ = {np.degrees(np.angle(np.exp(1j * beta))) + 0.0:.0f}°:  "
                  f"D = {10 * np.log10(D):.1f} dBi, HPBW = {hp:.1f}°", fontsize=10)
    ax2.grid(alpha=0.3); ax2.legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.show()
    lim = ant.grating_lobe_free_spacing(th0)
    print(f"grating-lobe-free spacing at this steering angle: d/lambda < {lim:.2f}"
          + ("   -> grating lobe(s) visible!" if d_over_lambda >= lim else ""))

interact(show_array, N=IntSlider(value=8, min=1, max=32), d_over_lambda=FloatSlider(value=0.5, min=0.1, max=1.5, step=0.05), steer_deg=FloatSlider(value=90, min=0, max=180, step=5), element=Dropdown(options=["isotropic", "half-wave dipole (along z)", "small dipole (along z)"], value="isotropic"));

**Directivity grows with $N$.** For a broadside array of isotropic elements with $d=\lambda/2$ the directivity
is exactly $D = N$ and the beamwidth shrinks like $\mathrm{HPBW}\approx 0.886\,\lambda/(Nd)$ rad — the
numerical integration confirms both.

In [ ]:
Ns = np.array([2, 4, 8, 16, 32, 64])
t = np.linspace(0, np.pi, 40001)
Dn = [ant.directivity(lambda th, ph, n=n: ant.array_factor(th, n, 0.5) ** 2, n_theta=8001, n_phi=3) for n in Ns]
hp = [np.degrees(ant.hpbw(ant.array_factor(t, n, 0.5), t)) for n in Ns]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].semilogx(Ns, 10 * np.log10(Dn), "o", color="tab:blue", ms=8, label="numerical integration")
axes[0].semilogx(Ns, 10 * np.log10(Ns), "k--", lw=1, label=r"$D = N$")
axes[0].set_ylabel("directivity (dBi)"); axes[0].set_title(r"broadside ULA, $d=\lambda/2$")
axes[1].loglog(Ns, hp, "o", color="tab:orange", ms=8, label="numerical HPBW")
axes[1].loglog(Ns, np.degrees(0.886 / (Ns * 0.5)), "k--", lw=1, label=r"$0.886\,\lambda/(Nd)$")
axes[1].set_ylabel("HPBW (deg)"); axes[1].set_title("beamwidth")
axes[1].set_yticks([2, 5, 10, 20, 50]); axes[1].set_yticklabels(["2", "5", "10", "20", "50"])
axes[1].yaxis.set_minor_formatter(plt.NullFormatter())
for ax in axes:
    ax.set_xlabel("number of elements N"); ax.set_xticks(Ns); ax.set_xticklabels(Ns)
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 5. Reflection: Fresnel coefficients and the Brewster angle
When a wave traveling in free space hits a smooth, large ($\gg\lambda$) boundary with relative permittivity
$\epsilon_r$ (complex for lossy ground, $\epsilon_r - j60\sigma\lambda$), part is reflected with coefficient
$\Gamma$. With $\psi$ the **grazing angle** (measured from the surface, as in the two-ray geometry):
$$\Gamma_{\mathrm{TE}} = \frac{\sin\psi - \sqrt{\epsilon_r - \cos^2\psi}}{\sin\psi + \sqrt{\epsilon_r - \cos^2\psi}}\ \ \text{(horizontal pol.)},\qquad
\Gamma_{\mathrm{TM}} = \frac{\epsilon_r\sin\psi - \sqrt{\epsilon_r - \cos^2\psi}}{\epsilon_r\sin\psi + \sqrt{\epsilon_r - \cos^2\psi}}\ \ \text{(vertical pol.)}.$$
Both tend to $\Gamma = -1$ at grazing incidence — the assumption behind the two-ray model. For vertical (TM)
polarization the reflection vanishes at the **Brewster angle** $\sin\psi_B = 1/\sqrt{\epsilon_r+1}$ (equivalently
$\tan\theta_B = \sqrt{\epsilon_r}$ measured from the normal). With losses the null becomes a deep minimum.
A perfect conductor ($\sigma\to\infty$) reflects everything: $|\Gamma| = 1$ at every angle.

In [ ]:
def show_fresnel(eps_r=15.0, sigma=0.005, f_mhz=900.0):
    psi = np.radians(np.linspace(0.01, 90, 900))
    eps = ant.complex_permittivity(eps_r, sigma, f_mhz * 1e6)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for pol, c in [("TE", "tab:blue"), ("TM", "tab:orange")]:
        g = ant.fresnel_reflection(psi, eps, pol)
        lab = "TE / horizontal" if pol == "TE" else "TM / vertical"
        axes[0].plot(np.degrees(psi), np.abs(g), color=c, label=lab)
        axes[1].plot(np.degrees(psi), np.degrees(np.angle(g)), color=c, label=lab)
    psi_b = np.degrees(ant.brewster_angle(eps_r))
    for ax in axes:
        ax.axvline(psi_b, color="gray", ls=":", lw=1)
        ax.set_xlabel(r"grazing angle $\psi$ (deg from the surface)"); ax.grid(alpha=0.3); ax.set_xlim(0, 90)
    axes[0].text(psi_b + 1, 0.9, rf"Brewster $\psi_B$ = {psi_b:.1f}°" "\n" r"(lossless $\epsilon_r$)", fontsize=9)
    axes[0].set_ylabel(r"$|\Gamma|$"); axes[0].set_ylim(0, 1.02); axes[0].legend(loc="best")
    axes[1].set_ylabel(r"$\angle\Gamma$ (deg)"); axes[1].set_ylim(-185, 185); axes[1].set_yticks(range(-180, 181, 60))
    fig.suptitle(rf"$\epsilon_r$ = {eps_r:g}, $\sigma$ = {sigma:g} S/m, f = {f_mhz:g} MHz  "
                 rf"(complex $\epsilon_r$ = {eps.real:.1f} $-$ j{-eps.imag:.2f})")
    plt.tight_layout(); plt.show()

interact(show_fresnel, eps_r=FloatSlider(value=15, min=1.5, max=81, step=0.5), sigma=FloatLogSlider(value=0.005, base=10, min=-4, max=1, step=0.25), f_mhz=FloatLogSlider(value=900, base=10, min=1, max=4.5, step=0.1));

Typical values: dry ground $\epsilon_r\approx 4$, $\sigma\approx 10^{-3}$ S/m; average ground $\epsilon_r\approx 15$,
$\sigma\approx 5\times10^{-3}$ S/m; sea water $\epsilon_r\approx 81$, $\sigma\approx 4$ S/m. Try sea water at 1 MHz vs. 3 GHz:
the conductivity term $60\sigma\lambda$ dominates at low frequency and the surface behaves like a conductor.

## 6. The two-ray ground-reflection model
A direct ray (length $d_{\mathrm{LOS}}$) and a ground-reflected ray ($d_{\mathrm{ref}}$) add at the receiver:
$$\frac{P_{\mathrm{RX}}}{P_{\mathrm{TX}}G_{\mathrm{TX}}G_{\mathrm{RX}}} = \left(\frac{\lambda}{4\pi}\right)^2
\left|\frac{e^{-jk_0 d_{\mathrm{LOS}}}}{d_{\mathrm{LOS}}} + \Gamma(\psi)\,\frac{e^{-jk_0 d_{\mathrm{ref}}}}{d_{\mathrm{ref}}}\right|^2 .$$
With a flat, perfectly conducting earth ($\Gamma=-1$), $d_{\mathrm{ref}} - d_{\mathrm{LOS}} \approx 2h_bh_m/d$ and equal amplitudes, this is
the slide's
$$P_{\mathrm{RX}}(d) = 4P_{\mathrm{TX}}G_{\mathrm{TX}}G_{\mathrm{RX}}\left(\frac{\lambda}{4\pi d}\right)^2\sin^2\!\left(\frac{2\pi h_bh_m}{\lambda d}\right)
\ \xrightarrow{\ d\ \gg\ 2\pi h_bh_m/\lambda\ }\ P_{\mathrm{TX}}G_{\mathrm{TX}}G_{\mathrm{RX}}\frac{h_b^2h_m^2}{d^4}.$$
The last peak of the $\sin^2$ (the **breakpoint** or critical distance) is at $d_c = 4h_bh_m/\lambda$. Before it the
two rays alternately add (+6 dB over free space) and cancel; well beyond it the power falls 40 dB/decade
and no longer depends on frequency. The free-space and $d^{-4}$ asymptotes cross at $4\pi h_bh_m/\lambda = \pi d_c$.

In [ ]:
def show_two_ray(h_b=30.0, h_m=1.5, f_mhz=900.0, ground="perfect conductor (Gamma = -1)"):
    f = f_mhz * 1e6
    d = np.logspace(1, 5, 4000)
    dc = ant.two_ray_breakpoint(f, h_b, h_m)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.semilogx(d, linear_to_db(ant.free_space_gain(d, f)), color="gray", lw=1.2, label=r"free space ($d^{-2}$)")
    if ground.startswith("perfect"):
        exact = ant.two_ray_gain(d, f, h_b, h_m, gamma=-1.0)
    else:
        pol = "TE" if "horizontal" in ground else "TM"
        exact = ant.two_ray_gain(d, f, h_b, h_m, eps_r=ant.complex_permittivity(15, 0.005, f), polarization=pol)
    ax.semilogx(d, linear_to_db(exact), color="tab:blue", lw=1.2, label=f"exact two-ray, {ground}")
    ax.semilogx(d, linear_to_db(ant.two_ray_gain_slide(d, f, h_b, h_m) + 1e-30), "--", color="tab:orange",
                lw=1.2, label=r"slide formula ($\sin^2$)")
    ax.semilogx(d, linear_to_db(ant.two_ray_gain_asymptotic(d, h_b, h_m)), ":", color="k", lw=1.5,
                label=r"$h_b^2h_m^2/d^4$ (40 dB/decade)")
    ax.axvline(dc, color="tab:red", ls="--", lw=1)
    ax.text(dc * 1.1, 0.97, rf"$d_c = 4h_bh_m/\lambda$ = {dc:,.0f} m", color="tab:red",
            transform=ax.get_xaxis_transform(), va="top", fontsize=9)
    ref = linear_to_db(ant.free_space_gain(d[-1], f))
    ax.set_ylim(min(ref, linear_to_db(ant.two_ray_gain_asymptotic(d[-1], h_b, h_m))) - 15, -20)
    ax.set_xlabel("distance d (m)"); ax.set_ylabel(r"$P_{\mathrm{RX}}/(P_{\mathrm{TX}}G_{\mathrm{TX}}G_{\mathrm{RX}})$ (dB)")
    ax.set_title(rf"two-ray model: $h_b$ = {h_b:g} m, $h_m$ = {h_m:g} m, f = {f_mhz:g} MHz")
    ax.grid(True, which="both", alpha=0.3); ax.legend(loc="lower left", fontsize=8); plt.show()
    far = d > 10 * dc
    if far.sum() > 10:
        slope = np.polyfit(np.log10(d[far]), linear_to_db(exact[far]), 1)[0]
        print(f"fitted slope beyond 10 d_c: {slope:.1f} dB/decade  (free space: -20, two-ray asymptote: -40)")

interact(show_two_ray, h_b=FloatSlider(value=30, min=2, max=100, step=1), h_m=FloatSlider(value=1.5, min=0.5, max=10, step=0.5), f_mhz=FloatLogSlider(value=900, base=10, min=2, max=4.5, step=0.05), ground=Dropdown(options=["perfect conductor (Gamma = -1)", "ground eps_r=15, horizontal pol.", "ground eps_r=15, vertical pol."], value="perfect conductor (Gamma = -1)"));

Frequency independence far out, checked numerically: at $d = 20$ km the $d^{-4}$ model predicts the same
path gain at every carrier, while free space loses 20 dB per decade of frequency.

In [ ]:
d, hb, hm = 20e3, 30.0, 1.5
print(f"{'f (MHz)':>8}{'d_c (km)':>10}{'exact two-ray':>15}{'h^2h^2/d^4':>12}{'free space':>12}   (all in dB)")
for fm in [150, 450, 900, 1900]:
    f = fm * 1e6
    print(f"{fm:8d}{ant.two_ray_breakpoint(f, hb, hm) / 1e3:10.2f}{linear_to_db(ant.two_ray_gain(d, f, hb, hm)):15.1f}"
          f"{linear_to_db(ant.two_ray_gain_asymptotic(d, hb, hm)):12.1f}{linear_to_db(ant.free_space_gain(d, f)):12.1f}")

At 20 km the link is 18–220 breakpoint distances out for all four carriers, so the exact model and the
$d^{-4}$ law agree to 0.1 dB. Note also in the plot above that the slide's $\sin^2$ formula overshoots close to the
base station ($d \lesssim h_b$): it assumes equal ray amplitudes and $d_{\mathrm{ref}}-d_{\mathrm{LOS}}\approx 2h_bh_m/d$, both of
which need $d \gg h_b, h_m$.

## 7. Knife-edge diffraction and Fresnel zones
By Huygens' principle, the wavefront above an obstruction re-radiates into the shadow region. For a single
knife edge that sits a height $h$ above the line of sight ($h<0$: below it), at distances $d_1$ and $d_2$ from
the terminals,
$$\nu = h\sqrt{\frac{2(d_1+d_2)}{\lambda d_1d_2}},\qquad
F(\nu) = \frac{E_d}{E_o} = \frac{1+j}{2}\int_\nu^\infty \exp\!\left(\frac{-j\pi t^2}{2}\right)dt,\qquad
G_d\,(\mathrm{dB}) = 20\log_{10}|F(\nu)| .$$
The integral is $\left[\tfrac12 - C(\nu)\right] - j\left[\tfrac12 - S(\nu)\right]$ with the Fresnel integrals $C, S$
(`scipy.special.fresnel`). Lee's piecewise approximation is often used instead:

| $\nu$ | $G_d$ (dB) |
|---|---|
| $\le -1$ | $0$ |
| $-1 \ldots 0$ | $20\log_{10}(0.5 - 0.62\nu)$ |
| $0 \ldots 1$ | $20\log_{10}(0.5\,e^{-0.95\nu})$ |
| $1 \ldots 2.4$ | $20\log_{10}\!\left(0.4 - \sqrt{0.1184 - (0.38 - 0.1\nu)^2}\right)$ |
| $> 2.4$ | $20\log_{10}(0.225/\nu)$ |

In [ ]:
nu = np.linspace(-3, 5, 1601)
exact = ant.knife_edge_gain_db(nu)
lee = ant.knife_edge_gain_lee_db(nu)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(nu, exact, color="tab:blue", label=r"exact $20\log_{10}|F(\nu)|$")
axes[0].plot(nu, lee, "--", color="tab:orange", label="Lee approximation")
axes[0].axhline(-6.02, color="gray", ls=":", lw=1); axes[0].axvline(0, color="gray", ls=":", lw=1)
axes[0].annotate(r"grazing ($\nu = 0$): $-6$ dB", xy=(0, -6.02), xytext=(0.6, -2.5), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="gray"))
axes[0].set_xlabel(r"Fresnel–Kirchhoff parameter $\nu$"); axes[0].set_ylabel(r"diffraction gain $G_d$ (dB)")
axes[0].set_title("knife-edge diffraction gain"); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].plot(nu, lee - exact, color="tab:orange")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlabel(r"$\nu$"); axes[1].set_ylabel("Lee $-$ exact (dB)")
axes[1].set_title("approximation error (Lee ignores the ripple for $\\nu < -1$)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"max gain {exact.max():.2f} dB at nu = {nu[np.argmax(exact)]:.2f};  "
      f"max |Lee - exact| for nu in [-1, 5]: {np.abs(lee - exact)[nu >= -1].max():.2f} dB")

**Fresnel zones.** The $n$-th Fresnel zone is the ellipsoid of points whose excess path over the direct ray
is $n\lambda/2$; its radius at distances $d_1,d_2$ is
$$r_n = \sqrt{\frac{n\lambda d_1 d_2}{d_1+d_2}},\qquad\text{so}\qquad \nu = \sqrt{2}\,\frac{h}{r_1}.$$
An obstruction that leaves the first Fresnel zone mostly clear costs little: $h = -0.6\,r_1$ gives $\nu \approx -0.85$
and $G_d \approx 0$ dB, the common "60 % clearance" rule for point-to-point links. Rappaport's Example 4.7
($\lambda = 1/3$ m, $d_1 = d_2 = 1$ km, $h = 25$ m) gives $\nu \approx 2.74$ and $G_d \approx -22$ dB: try it below.

In [ ]:
def show_knife_edge(h=25.0, d1_km=1.0, d2_km=1.0, f_mhz=900.0):
    lam = wavelength(f_mhz * 1e6)
    d1, d2 = d1_km * 1e3, d2_km * 1e3
    nu_ = ant.fresnel_kirchhoff_parameter(h, d1, d2, lam)
    gd = ant.knife_edge_gain_db(nu_)
    gl = ant.knife_edge_gain_lee_db(np.array([nu_]))[0]
    x = np.linspace(0, d1 + d2, 600)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1.6, 1]})
    ax = axes[0]
    for n, c in [(1, "tab:blue"), (2, "tab:green"), (3, "tab:purple")]:
        r = ant.fresnel_zone_radius(n, x, d1 + d2 - x, lam)
        ax.fill_between(x / 1e3, -r, r, color=c, alpha=0.12 if n > 1 else 0.25, lw=0)
        ax.plot(x / 1e3, r, color=c, lw=1, label=f"Fresnel zone {n}")
        ax.plot(x / 1e3, -r, color=c, lw=1)
    ax.plot([0, (d1 + d2) / 1e3], [0, 0], "k-", lw=1, label="line of sight")
    ymin = min(-1.6 * ant.fresnel_zone_radius(3, d1, d2, lam), h - 5)
    ax.fill_between([d1 / 1e3 - 0.004 * (d1 + d2) / 1e3, d1 / 1e3 + 0.004 * (d1 + d2) / 1e3], ymin, h,
                    color="0.25", label=f"knife edge (h = {h:g} m)")
    r1 = ant.fresnel_zone_radius(1, d1, d2, lam)
    ax.set_ylim(ymin, max(1.6 * ant.fresnel_zone_radius(3, d1, d2, lam), h + 5))
    ax.set_xlabel("distance along path (km)"); ax.set_ylabel("height above LOS (m)")
    ax.set_title(rf"$r_1$ = {r1:.1f} m at the edge,  $h/r_1$ = {h / r1:.2f},  $\nu$ = {nu_:.2f}", fontsize=10)
    ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)
    nn = np.linspace(-3, 5, 801)
    axes[1].plot(nn, ant.knife_edge_gain_db(nn), color="tab:blue", label="exact")
    axes[1].plot(nn, ant.knife_edge_gain_lee_db(nn), "--", color="tab:orange", label="Lee")
    axes[1].plot(nu_, gd, "o", color="tab:red", ms=9, label=rf"$G_d$ = {gd:.1f} dB (Lee {gl:.1f} dB)")
    axes[1].set_xlim(-3, 5); axes[1].set_ylim(-30, 3)
    axes[1].set_xlabel(r"$\nu$"); axes[1].set_ylabel(r"$G_d$ (dB)"); axes[1].grid(alpha=0.3)
    axes[1].legend(loc="lower left", fontsize=8); axes[1].set_title(f"f = {f_mhz:g} MHz", fontsize=10)
    plt.tight_layout(); plt.show()

interact(show_knife_edge, h=FloatSlider(value=25, min=-40, max=40, step=1), d1_km=FloatSlider(value=1, min=0.1, max=5, step=0.1), d2_km=FloatSlider(value=1, min=0.1, max=5, step=0.1), f_mhz=FloatLogSlider(value=900, base=10, min=2, max=4.5, step=0.05));

## Try it yourself
* A 60 GHz base-station array is 10 cm across. Where does its far field start, and would a user 2 m away be
  in the near field? What does that imply for beam training that assumes plane waves?
* Using the array demo, steer an 8-element, $d = 0.7\lambda$ array from broadside toward endfire. At what steering
  angle does the first grating lobe appear, and does it match $d/\lambda < 1/(1+|\cos\theta_0|)$? Why do
  mmWave arrays use $d \le \lambda/2$?
* In the two-ray demo switch to vertical polarization over real ground and move the base station up and down.
  Where does the Brewster angle show up in the received power, and why does the $d^{-4}$ region survive anyway?
* A 3.5 GHz link spans 4 km with a ridge 1 km from one end. How high above the line of sight may the ridge be
  (or how far below it must it be) for less than 3 dB diffraction loss? Compare the exact result with Lee's formula.